# Reproducing Table 5 of the NRP-Exp Paper

This notebook reproduces the illustrative workflow reported in Table 5 of the paper.

Configuration:

- Instance: `Concise_Case_11`
- Variant: `Excl(r7)`
- Objectives:
  - Satisfaction (maximize)
  - Effort (minimize)
- Constraint:
  - Effort ≤ 28
- Solver:
  - Brute Force
- Parameter:
  - max_evaluations = 30000

The notebook executes exactly the workflow reported in the paper and computes the corresponding quality indicators.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate project root."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

## Import Framework Components

Import the components required to reproduce the workflow.

In [3]:
import json
import time
import pandas as pd

from src.domain.core.loader import GenericJSONLoader
from src.domain.core.pipeline import PreprocessingEngine
from src.domain.plugins.nrp import NRPPlugin

from src.solving.algorithms.brute_force import BruteForceSolver

from src.solving.core.problem import (
    OptimizationProblem,
    ConstraintSpec,
    ThresholdOperator,
)

ModuleNotFoundError: No module named 'src'

## Load the Original Instance

Load the `Concise_Case_11` instance distributed with NRP-Exp.

In [ ]:
INSTANCE_PATH = (
    PROJECT_ROOT
    / "examples"
    / "instances"
    / "paper"
    / "Concise_Case_11.json"
)

raw_json = INSTANCE_PATH.read_text(
    encoding="utf-8"
)

base_model = GenericJSONLoader.load(
    raw_json,
    NRPPlugin,
)

print(base_model.name)

## Execute Preprocessing

Apply aggregation, coupling resolution, value-dependency processing, and exclusion branching.

In [ ]:
processed_models = PreprocessingEngine.process(
    initial_model=base_model,
    pipeline_options={}
)

print(
    "Generated variants:",
    len(processed_models)
)

## Inspect Generated Variants

The paper uses the variant corresponding to `Excl(r7)`.

In [ ]:
for idx, model in enumerate(processed_models):
    print(
        idx,
        getattr(
            model,
            "branch_name",
            f"Variant {idx}"
        ),
        len(model.items)
    )

## Select the Variant Reported in Table 5

In [ ]:
SELECTED_VARIANT_NAME = "Excl(r7)"

matching_models = [
    candidate
    for candidate in processed_models
    if getattr(
        candidate,
        "branch_name",
        "",
    ) == SELECTED_VARIANT_NAME
]


if len(matching_models) != 1:
    raise ValueError(
        f"Expected one variant named "
        f"{SELECTED_VARIANT_NAME!r}, "
        f"but found {len(matching_models)}."
    )

model = matching_models[0]


print(
    getattr(
        model,
        "branch_name",
        "Variant"
    )
)

## Configure Objectives and Constraints

Table 5 uses:

- Satisfaction (max)
- Effort (min)
- Effort ≤ 28

In [ ]:
OBJECTIVES = {
    "satisfaction": "max",
    "effort": "min",
}

constraints = {
    "effort": ConstraintSpec(
        attribute="effort",
        operator=ThresholdOperator.LESS_EQUAL,
        value=28.0,
    )
}

problem = OptimizationProblem(
    model=model,
    objectives=OBJECTIVES,
    constraints=constraints,
)

## Execute  Brute Force

Configuration reported in Table 5:

- max_evaluations = 30000

In [ ]:
solver = BruteForceSolver(
    max_evaluations=30000
)

start = time.perf_counter()

pareto_front = solver.solve(problem)

elapsed = (
    time.perf_counter() - start
)

pareto_front.set_objective_directions(
    problem.get_objective_directions()
)

print(
    "Execution time:",
    elapsed
)

print(
    "Solutions:",
    len(pareto_front.solutions)
)

## Inspect the Returned Front

In [ ]:
pareto_df = pareto_front.to_dataframe()

pareto_df.head(20)

## Compute Quality Indicators

Compute the indicators reported in Table 5.

In [ ]:
objective_vectors = {
    (
        round(
            float(
                solution.objectives["satisfaction"]
            ),
            6,
        ),
        round(
            float(
                solution.objectives["effort"]
            ),
            6,
        ),
    )
    for solution
    in pareto_front.solutions
}

print(
    "Plans:",
    len(pareto_front.solutions)
)

print(
    "Objective vectors:",
    len(objective_vectors)
)


metrics = pareto_front.compute_metrics(
    objective_names=[
        "satisfaction",
        "effort",
    ],
    directions={
        "satisfaction": "max",
        "effort": "min",
    }
)

metrics

## Table 5 Summary

Generate a summary directly comparable to the values reported in the paper.

In [ ]:
summary = {
    "Instance": "Concise_Case_11",
    "Variant": getattr(
        model,
        "branch_name",
        "Excl(r7)"
    ),
    "Constraint": "Effort <= 28",
    "Solver": "Brute Force",
    "max_evaluations": 30000,
    "Solutions": len(
        pareto_front.solutions
    ),
    "Objective vectors": len(
        objective_vectors
    ),
    **metrics,
}

pd.Series(summary)

## Reproduction Completed

This notebook reproduces the workflow reported in Table 5:

- Concise_Case_11
- Excl(r7)
- Satisfaction (max)
- Effort (min)
- Effort ≤ 28
- Brute Force
- max_evaluations = 30000

The generated outputs can be compared directly against the values reported in the manuscript.